# Gestión del Riesgo de Desastres — Análisis Estadístico de Precipitación

**Bloque:** A — Gestión | **Línea temática:** `gestion_riesgo` | **Variable principal:** precipitación acumulada (mm/mes)

---

## ¿Qué es la Gestión del Riesgo de Desastres (GRD)?

La GRD en Colombia es el proceso social orientado a conocer, reducir y manejar las condiciones de riesgo de desastres, con miras a la seguridad, el bienestar y la calidad de vida de las personas. Está regulada por la **Ley 1523 de 2012**, que creó el **Sistema Nacional de Gestión del Riesgo de Desastres (SNGRD)** y estableció los planes institucionales obligatorios.

El análisis estadístico de la precipitación es la piedra angular de la gestión del riesgo hidrometeorológico: inundaciones, avenidas torrenciales y movimientos en masa tienen a la lluvia como factor detonante principal. Sin un modelo robusto de la variabilidad pluviométrica —incluyendo tendencias, estacionalidad y la influencia del ENSO— no es posible calibrar los umbrales de alerta, dimensionar obras de mitigación ni priorizar inversión en el PMGRD municipal.

## Instituciones clave

| Institución | Rol principal |
|---|---|
| **UNGRD** | Coordinación del SNGRD, política nacional de riesgo |
| **IDEAM** | Datos hidroclimatológicos, mapas de amenaza, SPI |
| **SGC** | Amenaza sísmica, volcánica y movimientos en masa |
| **IGAC** | Cartografía base, catastro, análisis de pendientes |
| **CARs** | GRD a escala de cuenca (POMCA), determinantes ambientales |
| **Municipios** | PMGRD — Planes Municipales de Gestión del Riesgo |

## Preguntas que responde este notebook

1. ¿Existen tendencias estadísticamente significativas en la frecuencia e intensidad de eventos extremos de precipitación?
2. ¿Cuántos meses en el período analizado superaron el umbral de alerta de 300 mm/mes asociado a riesgo de inundación?
3. ¿Qué patrones estacionales domina el régimen pluviométrico? (temporadas húmedas MAM y SON)
4. ¿En qué medida el ENSO (con lag de 3 meses) explica la variabilidad interanual de la precipitación?
5. ¿Qué modelo predictivo —SARIMA, SARIMAX con covariables climáticas, XGBoost o Random Forest— ofrece mejor rendimiento para proyectar precipitación a 6 meses?

> **Ficha de dominio completa:** [`docs/fuentes/gestion_riesgo.md`](../../docs/fuentes/gestion_riesgo.md)  
> **Decisiones metodológicas:** [`docs/decisiones.md`](../../docs/decisiones.md)

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "gestion_riesgo"
VARIABLE = "precipitacion"
UNIDAD = "mm"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `gestion_riesgo` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "gestion_riesgo.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos

### Fuentes de datos oficiales para precipitación en Colombia

Los datos de precipitación para análisis de riesgo provienen principalmente de la red de estaciones hidrometeorológicas del IDEAM, que opera más de 3.500 puntos de monitoreo en el país. Las fuentes alternativas complementarias incluyen:

| Fuente | Tipo | Resolución | Acceso |
|---|---|---|---|
| **IDEAM DHIME** | Estaciones in situ | Diaria / Mensual | [dhime.ideam.gov.co](http://dhime.ideam.gov.co) |
| **ERA5 (Copernicus)** | Reanálisis global | Horaria, 0.25° | Libre (CDS API) |
| **CHIRPS** | Satélite + gauge | Diaria, 0.05° | Libre (CHC) |
| **TRMM / GPM IMERG** | Satélite NASA | 30 min, 0.1° | Libre (NASA GES DISC) |

**Nota sobre la variable:** La precipitación acumulada mensual (mm/mes) es la agregación estándar para análisis de riesgo de inundación. El umbral operativo de **300 mm/mes** es la referencia del ENA 2022 para indicar condiciones de riesgo elevado de inundación en planicies aluviales colombianas.

> Para conectar directamente con DHIME: `from estadistica_ambiental.io.connectors import load_ideam_dhime`  
> Colocar el archivo en `data/raw/` y ajustar la ruta abajo.

In [ ]:
# df = load_csv("data/raw/gestion_riesgo.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "precipitacion": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 1b. Componentes AVR — Amenaza, Vulnerabilidad y Riesgo

La **Ley 1523/2012** define el riesgo como la combinacion de amenaza y vulnerabilidad:

```
Riesgo = Amenaza x Vulnerabilidad x Exposicion
Escala: Muy Baja (1) | Baja (2) | Media (3) | Alta (4) | Muy Alta (5)
```

| Fenomeno | Indicador clave | Detonante | Fuente |
|---|---|---|---|
| Inundacion | Profundidad agua (m), caudal | Lluvia extrema | IDEAM/HEC-RAS |
| Movimiento masa | Pendiente, litologia | Lluvia/sismo | SGC |
| Avenida torrencial | IVET (0-1) | Lluvia intensa | SGC/IDEAM |
| Sequia | SPI < -1.5 | Deficit lluvia | IDEAM |

**IVET (Indice de Vulnerabilidad a Eventos Torrenciales):** combina pendiente, cobertura, morfometria de cuenca e indice de Melton. IVET > 0.6 = alta susceptibilidad.

**SPI (Standardized Precipitation Index):** SPI < -1.5 = sequia severa; SPI > 1.5 = lluvia extrema.

In [ ]:
# AVR sintetico por municipio + SPI de precipitacion
np.random.seed(123)
n = len(df)

# -- SPI (Indice de Precipitacion Estandarizado) -------------------------
precip = df['precipitacion'].values
mu_p, std_p = precip.mean(), precip.std()
spi = (precip - mu_p) / std_p  # SPI-1 mensual
df['spi'] = spi.round(3)

# -- IVET por subcuenca (valor anual, escala 0-1) -------------------------
n_subcuencas = 8
subcuencas = [f'SC-{i+1:02d}' for i in range(n_subcuencas)]
ivet = np.random.beta(2, 3, n_subcuencas)  # distribucion tipica IVET

# -- Indice de Amenaza (IA) + Vulnerabilidad (IV) + Riesgo (IR) ----------
# Escala 1-5, simulando evaluacion AVR por municipio
n_municipios = 15
municipios = [f'Mpio-{i+1:02d}' for i in range(n_municipios)]
ia = np.random.randint(1, 6, n_municipios)
iv = np.random.randint(1, 6, n_municipios)
ir = np.clip((ia * iv / 5).astype(int), 1, 5)

df_avr = pd.DataFrame({'municipio': municipios, 'amenaza': ia,
                        'vulnerabilidad': iv, 'riesgo': ir})
df_avr['categoria_riesgo'] = pd.cut(df_avr['riesgo'],
    bins=[0,1,2,3,4,5], labels=['Muy Baja','Baja','Media','Alta','Muy Alta'])

print(f'SPI | min={spi.min():.2f} max={spi.max():.2f}')
print(f'Meses SPI < -1.5 (sequia severa): {(spi < -1.5).sum()}')
print(f'Meses SPI > 1.5 (lluvia extrema): {(spi > 1.5).sum()}')
print(f'\nIVET subcuencas | max={ivet.max():.3f} (alta susceptibilidad torrent si >0.6)')
print(f'Subcuencas IVET > 0.6: {(ivet > 0.6).sum()}/{n_subcuencas}')
df_avr

## 2. Validación y EDA

### Por qué validar antes de analizar

En datos de riesgo, los errores de ingesta pueden ser costosos: un pico espurio de precipitación puede inflar artificialmente la frecuencia de excedencias y llevar a sub o sobreestimación del riesgo. La función `validate()` aplica rangos físicos específicos configurados en `config.py` para la línea `gestion_riesgo`:

- **Precipitación:** rango físico 0–7000 mm/mes (máximos históricos en el Pacífico colombiano superan 1000 mm/mes).
- Detecta saltos temporales (lagunas en la serie), duplicados de fecha y outliers extremos.
- **Importante (ADR-002):** Los picos de precipitación reales NO se eliminan automáticamente. Son señal de eventos de amenaza. El módulo `preprocessing/outliers.py` es opt-in explícito.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_gestion_riesgo.html",
       title="EDA — Gestión de Riesgo", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

### Qué buscar en el análisis visual

La visualización temporal de precipitación revela la estructura que guía todas las decisiones de modelado posteriores:

- **Tendencia:** ¿Aumenta o disminuye la precipitación en el largo plazo? (relevante para proyecciones climáticas y actualización de PMGRD)
- **Estacionalidad:** Colombia tiene un régimen bimodal en la mayor parte del territorio, con temporadas húmedas en **MAM (marzo–abril–mayo)** y **SON (septiembre–octubre–noviembre)**, y períodos secos en DEF y JJA. En el Pacífico el régimen es monomodal.
- **Outliers y eventos extremos:** Picos que superan 300 mm/mes (umbral ENA 2022) son eventos de interés para el análisis de excedencias. No eliminar sin justificación técnica (ADR-002).
- **Ruptura de tendencia:** El fenómeno de La Niña 2010–2011 generó la mayor ola invernal en la historia reciente de Colombia, con más de 3.2 millones de afectados. Detectar cambios estructurales es crítico.

In [ ]:
plot_series(df, "fecha", "precipitacion", title="Gestión de Riesgo — precipitacion (mm)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "precipitacion", period="month")
plt.show()

## 3c. Dashboard AVR — Visualizacion de riesgo compuesto

Los cuatro paneles responden preguntas clave del ciclo GRD (Ley 1523/2012):

- **SPI:** deteccion de sequias extremas (detonante hidrologico)
- **IVET:** susceptibilidad a avenidas torrenciales por subcuenca
- **Matriz AVR:** distribucion amenaza vs. vulnerabilidad por municipio
- **Riesgo compuesto:** clasificacion final para POT y POMCA

> **InSAR / DInSAR** (Interferometric SAR): complementa el AVR midiendo deformacion del terreno en milimetros, fundamental para monitorear movimientos en masa activos y zonas de subsidencia. Fuentes: ESA Sentinel-1 (C-SAR) o ALOS-2 PALSAR-2.

In [ ]:
RISK_COLORS = {'Muy Baja':'#27ae60','Baja':'#82e0aa','Media':'#f1c40f',
               'Alta':'#e67e22','Muy Alta':'#e74c3c'}

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Panel A: SPI mensual
ax = axes[0, 0]
colors_spi = ['#e74c3c' if s < -1.5 else '#e67e22' if s < 0
              else '#27ae60' if s > 1.5 else '#82e0aa' for s in df['spi']]
ax.bar(df['fecha'], df['spi'], color=colors_spi, width=20)
ax.axhline(-1.5, color='red', ls='--', lw=1.5, label='Sequia severa SPI<-1.5')
ax.axhline(1.5, color='blue', ls='--', lw=1.5, label='Lluvia extrema SPI>1.5')
ax.set_title('SPI — Indice de Precipitacion Estandarizado', fontweight='bold')
ax.set_ylabel('SPI'); ax.legend(fontsize=7); ax.grid(alpha=0.3)

# Panel B: IVET por subcuenca
ax = axes[0, 1]
colors_ivet = ['#e74c3c' if v > 0.6 else '#e67e22' if v > 0.4 else '#27ae60'
               for v in ivet]
ax.barh(subcuencas, ivet, color=colors_ivet, alpha=0.85)
ax.axvline(0.6, color='red', ls='--', lw=1.5, label='Alta susceptibilidad IVET>0.6')
ax.set_title('IVET — Susceptibilidad a Avenidas Torrenciales', fontweight='bold')
ax.set_xlabel('IVET (0-1)'); ax.legend(fontsize=7); ax.grid(axis='x', alpha=0.3)

# Panel C: Scatter Amenaza vs Vulnerabilidad
ax = axes[1, 0]
scatter_colors = df_avr['categoria_riesgo'].map(RISK_COLORS)
sc = ax.scatter(df_avr['amenaza'], df_avr['vulnerabilidad'],
                c=scatter_colors, s=80, alpha=0.8, edgecolors='gray', lw=0.5)
ax.set_xlabel('Amenaza (1-5)'); ax.set_ylabel('Vulnerabilidad (1-5)')
ax.set_title('Matriz Amenaza vs. Vulnerabilidad por Municipio', fontweight='bold')
for _, r in df_avr.iterrows():
    ax.annotate(r['municipio'], (r['amenaza'], r['vulnerabilidad']),
                fontsize=6, ha='center', va='bottom')
ax.grid(alpha=0.3)

# Panel D: Distribucion riesgo compuesto
ax = axes[1, 1]
riesgo_counts = df_avr['categoria_riesgo'].value_counts().reindex(
    ['Muy Baja','Baja','Media','Alta','Muy Alta'], fill_value=0)
ax.bar(riesgo_counts.index, riesgo_counts.values,
       color=[RISK_COLORS[k] for k in riesgo_counts.index], alpha=0.85)
ax.set_title('Distribucion Riesgo Compuesto AVR (Ley 1523/2012)', fontweight='bold')
ax.set_ylabel('N municipios'); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Dashboard Gestion del Riesgo — AVR + SPI + IVET',
             fontweight='bold', fontsize=12)
plt.tight_layout(); plt.show()

alto_riesgo = (df_avr['categoria_riesgo'].isin(['Alta','Muy Alta'])).sum()
print(f'Municipios en riesgo Alto/Muy Alto: {alto_riesgo}/{n_municipios}')
print(df_avr[df_avr['categoria_riesgo'].isin(['Alta','Muy Alta'])][['municipio','amenaza','vulnerabilidad','categoria_riesgo']])

## 3b. Covariable ENSO (ONI)

### Por qué incorporar ENSO con lag en gestión de riesgo

El fenómeno ENSO (El Niño–Oscilación del Sur) es el principal modulador de la variabilidad climática interanual en Colombia. Su efecto sobre la precipitación no es inmediato: existe un rezago hidrológico de aproximadamente **3 meses** entre el pico de la anomalía de temperatura superficial del mar (TSM) en el Pacífico tropical y su manifestación en los caudales y lluvias colombianas.

- **El Niño (ONI > +0.5°C):** Reduce la precipitación en el interior andino y Caribe; aumenta el riesgo de sequía e incendios forestales.
- **La Niña (ONI < −0.5°C):** Incrementa la precipitación; eleva el riesgo de inundaciones y movimientos en masa.
- **Lag de 3 meses para `gestion_riesgo`:** Definido en `config.ENSO_LAG_MESES` según literatura colombiana (IDEAM, ENA 2022).

> **ADR-007:** Siempre usar `enso_lagged()` en lugar de `enso_dummy()`. El lag se obtiene automáticamente de `config.ENSO_LAG_MESES` según la línea temática.

In [ ]:
# --- Covariable ENSO (lag específico para Gestión de Riesgo) ---
oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica="gestion_riesgo")
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva

### Qué interpretar en la descripción univariada de precipitación

La estadística descriptiva establece la línea base del régimen pluviométrico y permite detectar anomalías estructurales antes de modelar:

| Estadístico | Interpretación en contexto de riesgo |
|---|---|
| **Media** | Precipitación típica mensual; comparar con norma climática (período 1981–2010) |
| **Coeficiente de variación (CV)** | CV > 40% indica alta variabilidad interanual — mayor incertidumbre en la planificación |
| **Percentil 90 / 95** | Umbrales para clasificar eventos extremos según metodología del IDEAM |
| **Asimetría** | Distribuciones de precipitación son típicamente asimétricas positivas (gamma o log-normal) |
| **Máximo histórico** | Comparar con umbrales de alerta: >300 mm/mes (riesgo inundación, ENA 2022) |

**Distribución de referencia:** Para precipitación mensual en zonas andinas colombianas, la distribución Gamma con parámetros de forma k ≈ 2–5 y escala θ ≈ 30–80 mm es habitual. La distribución GEV (Gumbel, Fréchet, Weibull) se usa para valores extremos.

In [ ]:
summarize(df)

## 5. Inferencial

### Estacionariedad: por qué importa en series de precipitación

Una serie de precipitación no estacionaria (con tendencia o varianza cambiante) viola los supuestos de modelos ARIMA y puede producir pronósticos inconsistentes. La prueba dual **ADF + KPSS** es obligatoria (ADR-004):

- **ADF (Augmented Dickey-Fuller):** H₀ = raíz unitaria (no estacionaria). Si p < 0.05 → se rechaza H₀ → estacionaria.
- **KPSS:** H₀ = estacionaria. Si p < 0.05 → se rechaza H₀ → no estacionaria.
- **Decisión combinada:** Solo cuando ADF rechaza H₀ Y KPSS no rechaza H₀ se confirma estacionariedad. Si hay contradicción, aplicar diferenciación estacional (d=1, D=1) y re-evaluar.

**Comportamiento esperado en precipitación:** Series mensuales sin tendencia fuerte tienden a ser estacionarias (ADF: p < 0.05, KPSS: p > 0.05), pero la estacionalidad regular puede confundir la prueba. Verificar con ACF/PACF.

### Mann-Kendall: detectar tendencias en extremos

Además de la estacionariedad media, el test de **Mann-Kendall** (no paramétrico) detecta si existe una tendencia monótona en la serie — sin asumir normalidad. Es el estándar internacional para análisis de tendencia en hidrología:

- **slope (Sen's slope):** Cambio absoluto por unidad de tiempo (mm/mes por año). Una pendiente positiva indica intensificación del régimen pluvial.
- **p < 0.05:** Tendencia estadísticamente significativa — relevante para actualizar estudios de amenaza en los POT y POMCA.
- **Aplicación en extremos:** Se recomienda también aplicar Mann-Kendall a los **percentiles 90 y 95** de la serie para detectar cambios en la frecuencia de eventos extremos, no solo en la media.

In [ ]:
ts = df.set_index("fecha")["precipitacion"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5c. LSTM para prediccion de caudales extremos — arquitectura y flujo

Las **Redes Neuronales Recurrentes (LSTM — Long Short-Term Memory)** son el estado del arte para prediccion de series temporales hidrologicas porque capturan dependencias temporales largas (memoria de lluvia acumulada, saturacion del suelo).

```
Arquitectura tipica para GRD:
  Input: [precipitacion_t-k, ..., t-1, SPI, pendiente, cobertura]
  LSTM Layer 1: 64 unidades, dropout=0.2
  LSTM Layer 2: 32 unidades, dropout=0.2
  Dense: 1 salida (caudal_t o prob_deslizamiento)
  Loss: MSE (caudal) o BCE (clasificacion amenaza)
```

Librerias: `keras` / `tensorflow` o `pytorch`. Referencia IDEAM: modelos LSTM calibrados con datos DHIME para cuencas andinas (RMSE ~15% del caudal medio).

> **InSAR / DInSAR (Interferometric SAR):** complementa el LSTM midiendo deformacion del terreno en mm/ano. Sentinel-1 permite generar mapas de velocidad de subsidencia o desplazamiento. Critico para monitorear deslizamientos activos antes del colapso.

In [ ]:
# Simulacion LSTM-like: ventana deslizante para prediccion de caudal maximo
# (numpy puro — sin dependencias de tensorflow/pytorch)
WINDOW = 6   # meses de contexto
HORIZON = 1  # meses a predecir

ts_vals = df_clean['precipitacion'].values if 'precipitacion' in df_clean.columns else df_clean.iloc[:, 0].values

# Construir matrices X, y con ventana deslizante (como LSTM)
X_lstm, y_lstm = [], []
for t in range(WINDOW, len(ts_vals) - HORIZON):
    X_lstm.append(ts_vals[t-WINDOW:t])  # contexto
    y_lstm.append(ts_vals[t:t+HORIZON]) # objetivo
X_lstm = np.array(X_lstm)  # shape: (muestras, WINDOW)
y_lstm = np.array(y_lstm).ravel()

# Regresion lineal como proxy del LSTM (misma ventana deslizante)
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
ridge = Ridge(alpha=1.0)
cv_rmse = (-cross_val_score(ridge, X_lstm, y_lstm,
                            cv=5, scoring='neg_root_mean_squared_error')).mean()
ridge.fit(X_lstm, y_lstm)
y_pred = ridge.predict(X_lstm)

# Plot: prediccion vs real
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(y_lstm, lw=1.5, color='#2980b9', label='Real')
ax1.plot(y_pred, lw=1.5, ls='--', color='#e74c3c', label='LSTM-proxy (Ridge, ventana=6m)')
ax1.set_title(f'LSTM-proxy — Prediccion caudal | CV-RMSE={cv_rmse:.2f}', fontweight='bold')
ax1.set_xlabel('Paso temporal'); ax1.set_ylabel('Precipitacion (mm)')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

# SPI extremos
ax2.hist(df['spi'], bins=30, color='#3498db', alpha=0.7, edgecolor='white')
ax2.axvline(-1.5, color='red', ls='--', lw=2, label='Sequia severa')
ax2.axvline(1.5, color='green', ls='--', lw=2, label='Lluvia extrema')
ax2.set_title('Distribucion SPI — Deteccion de extremos hidrologicos', fontweight='bold')
ax2.set_xlabel('SPI'); ax2.set_ylabel('Frecuencia')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f'Ventana LSTM: {WINDOW} meses | CV-RMSE: {cv_rmse:.2f}')
print('Nota: LSTM real (TF/PyTorch) captura dependencias no lineales y memoria larga')
print('InSAR/DInSAR (Sentinel-1): monitoreo deformacion del terreno mm/ano complementa GRD')

## 5b. Análisis de excedencias normativas

### Umbrales de alerta para precipitación en Colombia

A diferencia de contaminantes del aire o calidad del agua, la precipitación no tiene normas de cumplimiento legal con valores fijos. Sin embargo, existen **umbrales operativos de alerta** usados por la UNGRD y los sistemas de alerta temprana (SAT) municipales:

| Umbral | Valor | Referencia | Acción típica |
|---|---|---|---|
| **Alerta amarilla** | 150–200 mm/mes | IDEAM / SAT municipal | Monitoreo intensificado |
| **Alerta naranja** | 200–300 mm/mes | IDEAM / SAT municipal | Alistamiento de equipos de respuesta |
| **Alerta roja (inundación)** | >300 mm/mes | ENA 2022 / UNGRD | Evacuación preventiva en zonas de amenaza alta |
| **Evento extremo percentil 95** | Variable por estación | IDEAM | Activación de PMGRD |

**Nota metodológica:** `exceedance_report()` busca automáticamente umbrales registrados en `config.py` para la variable. Para precipitación, si no hay umbral pre-configurado, se puede pasar `threshold=300` directamente para calcular la tasa de excedencia operativa.

**Análisis por temporada:** Comparar la frecuencia de excedencias en MAM vs SON vs períodos secos. Una temporada húmeda con excedencias frecuentes puede indicar acumulación de riesgo que justifica actualización del estudio de amenaza del POT.

In [ ]:
rep = exceedance_report(df["precipitacion"], variable="precipitacion")
if rep.empty:
    print("Sin normas colombianas registradas para 'precipitacion'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

### Tratamiento de datos faltantes en series de precipitación

Las series pluviométricas del IDEAM presentan frecuentemente datos faltantes por fallas de sensor, vandalismo de estaciones o errores en la transmisión. El tratamiento correcto depende del patrón de ausencia:

- **Faltantes aislados (< 5% de la serie):** Interpolación lineal o spline. Adecuado para períodos sin eventos extremos.
- **Bloques de ausencia (>1 mes continuo):** Imputar con el promedio climatológico del mismo mes en otros años, o usar datos de estaciones vecinas (kriging espacial).
- **Temporadas de evento (La Niña, El Niño):** Nunca imputar con promedios normales. El sesgo puede ser mayor que el dato faltante.

> **ADR-002:** No aplicar clipping (truncamiento) a valores extremos en precipitación. Los máximos son precisamente los eventos de amenaza que interesa estudiar.

El método `impute(..., method="linear")` es apropiado para faltantes aislados. Para bloques largos, considerar `method="seasonal_mean"` que respeta la climatología mensual.

In [ ]:
df_clean = impute(df.copy(), cols=["precipitacion"], method="linear")
print(f"Faltantes antes: {df["precipitacion"].isna().sum()} | "
      f"después: {df_clean["precipitacion"].isna().sum()}")

## 7. Modelos predictivos

### Selección de modelo para precipitación en gestión de riesgo

La elección del modelo depende del objetivo de uso del pronóstico y de la longitud y calidad de la serie histórica:

| Modelo | Ventajas | Cuándo preferirlo |
|---|---|---|
| **SARIMA** | Captura estacionalidad regular (S=12) sin covariables | Serie estacionaria, estacionalidad estable, sin ENSO significativo |
| **SARIMAX** | SARIMA + covariables exógenas (ONI, temperatura) | ENSO explica variabilidad interanual; mejora pronósticos en años de evento |
| **XGBoost** | Captura relaciones no lineales y regímenes mixtos | Serie larga (>5 años), múltiples covariables disponibles, outliers frecuentes |
| **Random Forest** | Robusto a outliers, fácil de interpretar | Mismas condiciones que XGBoost; buena opción baseline no paramétrico |

**Métricas de evaluación (ADR-003):**
- **RMSE:** Penaliza errores grandes — adecuado para precipitación donde los picos importan.
- **MAE:** Más robusto a outliers — usar como métrica secundaria.
- **sMAPE:** Para comparar entre estaciones con diferentes escalas de precipitación.
- **No usar RMSLE** para precipitación: aunque siempre es positiva, el log sesga la penalización hacia los valores bajos, minimizando el error en picos extremos que son los más relevantes para riesgo.

**Horizonte de pronóstico:** 6 meses es el estándar operativo para alertas de temporada. Para sistemas de alerta temprana (SAT) se requieren horizontes de 1–5 días con datos diarios.

In [ ]:
ts = df_clean.set_index("fecha")["precipitacion"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "SARIMAX":      get_model("sarimax", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
    "RandomForest": get_model("random_forest", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 8. Conclusiones

### Plantilla de conclusiones (completar con datos reales)

Al trabajar con datos de producción, documentar los siguientes hallazgos:

**Tendencia histórica:**
- Resultado Mann-Kendall: `[creciente / decreciente / sin tendencia]`, pendiente Sen = `X` mm/mes por año (p = `Y`).
- Interpretación: ¿Qué implicación tiene esta tendencia para la actualización del estudio de amenaza del municipio?

**Estacionalidad:**
- ¿La precipitación máxima ocurre consistentemente en temporadas MAM y SON?
- ¿El bimodalismo estacional es estable o ha cambiado en los últimos años?

**Excedencias normativas:**
- Número de meses con precipitación > 300 mm: `N` de `T` meses totales (`X%` del período).
- ¿Se concentran en temporadas húmedas o hay excedencias en períodos atípicos?

**Influencia ENSO:**
- Correlación entre ONI (lag 3 meses) y precipitación: r = `X`, p = `Y`.
- ¿El SARIMAX mejora significativamente el RMSE respecto al SARIMA puro?

**Mejor modelo:**
- Modelo seleccionado: `[nombre]` con RMSE = `X`, MAE = `Y`, sMAPE = `Z%`.

### Normativa y referencias

- **Ley 1523/2012:** Marco institucional del SNGRD — obligatoriedad de PMGRD.
- **Decreto 1807/2014:** Incorporación técnica de la GRD en los POT (estudios básicos y detallados).
- **ENA 2022 (IDEAM):** Umbral de 300 mm/mes como referencia de riesgo de inundación.
- Registrar decisiones metodológicas en [`docs/decisiones.md`](../../docs/decisiones.md).

## 9. Cómo adaptar a datos reales

### Paso 1: Obtener datos de precipitación de producción

```python
# Opción A: IDEAM DHIME (requiere credenciales institucionales)
from estadistica_ambiental.io.connectors import load_ideam_dhime
df = load_ideam_dhime(
    station_code="21205010",   # Código de la estación IDEAM
    variable="precipitacion",
    start="2000-01-01",
    end="2024-12-31",
    freq="ME"                  # Mensual
)

# Opción B: Archivo CSV exportado de DHIME
df = load_csv("data/raw/precipitacion_estacion_21205010.csv", date_col="fecha")
```

### Paso 2: Ajustar la variable y unidad

Cambiar en la celda de Setup:
```python
VARIABLE = "precipitacion"   # columna en el DataFrame
UNIDAD = "mm"
```

### Paso 3: Verificar rangos físicos esperados

Para Colombia, rangos de precipitación mensual por región:

| Región | Mínimo típico (mm/mes) | Máximo típico (mm/mes) |
|---|---|---|
| Caribe (seco) | 0 | 250 |
| Andina central | 30 | 450 |
| Pacífico (Chocó) | 100 | 1500+ |
| Orinoquía | 0 | 600 |
| Amazonía | 50 | 700 |

Si los valores de tu estación quedan fuera de estos rangos → revisar unidades (¿mm/día vs mm/mes?) o errores de digitación.

### Paso 4: Análisis de excedencias con umbral real

```python
from estadistica_ambiental.inference.intervals import exceedance_report
# Usar umbral operativo del SAT del municipio o la referencia ENA 2022
rep = exceedance_report(df["precipitacion"], variable="precipitacion", threshold=300)
print(f"Meses en alerta roja: {rep['n_exceedances'].sum()} / {len(df)}")
```

### Paso 5: Incorporar múltiples estaciones (análisis espacial)

Para el PMGRD municipal es útil analizar varias estaciones simultáneamente:
```python
estaciones = ["21205010", "21206020", "21207030"]
dfs = [load_ideam_dhime(code, "precipitacion") for code in estaciones]
# Comparar tendencias Mann-Kendall por estación
for i, d in enumerate(dfs):
    ts = d.set_index("fecha")["precipitacion"]
    mk = mann_kendall(ts)
    print(f"Estación {estaciones[i]}: {mk['trend']} | slope={mk['slope']:.4f}")
```

### Paso 6: Registrar decisiones en docs/decisiones.md

Al finalizar el análisis con datos reales, documentar:
- Estación(es) utilizadas y período de análisis
- Método de imputación aplicado y justificación
- Umbral de excedencia elegido y referencia normativa
- Modelo seleccionado y métricas de validación obtenidas
- Si se detectó tendencia significativa, implicación para actualización del PMGRD